# Click-ranking baseline

Compare random, freshness, user-category affinity, and logistic regression. All candidates in each sampled slate are retained. See `README.md` in this folder for methodology and run instructions.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

HERE = Path.cwd()
BASELINE_DIR = HERE if (HERE / "train_baseline.py").exists() else HERE / "Baseline Testing"
assert (BASELINE_DIR / "train_baseline.py").exists(), "Open from the project root or Baseline Testing folder."
PROJECT_ROOT = BASELINE_DIR.parent
DATA_CANDIDATES = [PROJECT_ROOT / "data", PROJECT_ROOT, PROJECT_ROOT.parent / "project"]
DATA_ROOT = next((p for p in DATA_CANDIDATES if (p / "articles.parquet").exists()), PROJECT_ROOT / "data")
# Change DATA_ROOT above or pass --root to the script if the dataset moves.
OUT = BASELINE_DIR / "results"


## 1. Train and evaluate

The saved benchmark is loaded by default. Set `RUN_TRAINING = True` to reproduce it. If no report exists, training runs automatically. The script scans source files in batches and samples whole impressions uniformly. If dependencies are missing, install `requirements.txt` in this folder into the notebook kernel environment.


In [ ]:
RUN_TRAINING = False
TRAIN_IMPRESSIONS = 50_000
VALIDATION_IMPRESSIONS = 10_000

if RUN_TRAINING or not (OUT / "report.json").exists():
    subprocess.run([
        sys.executable, str(BASELINE_DIR / "train_baseline.py"),
        "--root", str(DATA_ROOT), "--output", str(OUT),
        "--train-impressions", str(TRAIN_IMPRESSIONS),
        "--validation-impressions", str(VALIDATION_IMPRESSIONS),
        "--seed", "42",
    ], check=True)
else:
    print("Loading saved benchmark. Set RUN_TRAINING = True to retrain.")


## 2. Compare ranking quality

Mean AUC is computed within impressions. nDCG measures where observed clicked articles rank. Tied scores receive expected credit. Validation uses sampled impressions with complete candidate slates, not every impression in the source file.


In [3]:
metrics = pd.read_csv(OUT / "metrics.csv", index_col=0)
display(metrics[["mean_impression_auc", "ndcg@5", "ndcg@10",
                 "older_than_24h_share@5", "older_than_24h_share@10"]].round(4))


,mean_impression_auc,ndcg@5,ndcg@10,older_than_24h_share@5,older_than_24h_share@10
random_expected,0.5000,0.3469,0.4313,0.2485,0.2582
freshness,0.5038,0.3384,0.4226,0.0953,0.1693
category_affinity,0.5595,0.3968,0.4716,0.2319,0.2461
logistic_regression,0.7073,0.5395,0.5849,0.0988,0.1721


## 3. Check sample sizes, runtime, and metadata


In [4]:
report = json.loads((OUT / "report.json").read_text())
display(pd.DataFrame(report["splits"]).T)
print("Fit seconds:", round(report["fit_seconds"], 2))
print("Peak process memory (MB):", round(report["peak_rss_megabytes"], 1))
for note in report["notes"]:
    print("-", note)


,impressions,candidate_pairs,time_min,time_max,click_rate_in_sampled_slates,history_cutoff_exclusive,history_events_excluded_at_or_after_cutoff_or_missing_time,missing_article_pairs,missing_age_pairs,future_publication_pairs,feature_table_megabytes
train,50000,554696,2023-05-18 07:00:02,2023-05-25 06:59:59,0.090484,2023-05-18 07:00:02,0,0,0,129,82.241603
validation,10000,118731,2023-05-25 07:02:58,2023-06-01 06:59:14,0.08451,2023-05-25 07:02:58,0,0,0,2,17.609337


Fit seconds: 8.06
Peak process memory (MB): 1004.6
- Evaluation uses whole sampled validation slates, not the full validation population unless sample size is 0.
- No model selection or hyperparameter tuning on the supplied validation period.
- Probabilities refer to the supplied logged candidate population; they are not site-wide CTR estimates.
- History snapshots are fixed before the earliest sampled impression of each split.
- No candidate age cutoff, post-impression features, catalog totals, or candidate-list position features.
- Session continuity and diversity are deferred until a click-ranking baseline is established.
- The recorded run reused the exact saved uniform samples after optimizing history processing. Total runtime excludes regenerating those samples from the full source files.


## 4. Inspect one recommendation slate

The true click labels are for evaluation only. Shared ranks indicate tied predicted scores.


In [5]:
predictions = pd.read_parquet(OUT / "validation_predictions.parquet")
example_id = predictions.impression_id.iloc[0]
display(predictions.loc[predictions.impression_id == example_id]
        .sort_values("rank").head(10))


,impression_id,candidate_article_id,article_age_hours,clicked,click_probability,rank
0,343268098,9779867,10.683333,0,0.225255,1
1,343268098,9779748,12.412500,0,0.191800,2
4,343268098,9779777,10.239722,0,0.178038,3
10,343268098,9779427,9.625000,0,0.176736,4
9,343268098,9778939,14.982500,0,0.120403,5
14,343268098,9779860,10.653611,0,0.109661,6
11,343268098,9779648,9.948611,0,0.105164,7
13,343268098,9780020,3.686389,1,0.103441,8
12,343268098,9780193,1.902778,0,0.096602,9
5,343268098,9780271,0.635833,0,0.094210,10


## Next experiments

Use this benchmark before adding richer features or models. Tune future changes on a chronological development split inside training, then compare on the supplied validation period. Add continuation only if candidate-specific information improves its prediction beyond shared context. Test freshness and diversity separately, and retain an age-aware evaluation. Offline clicks and continuation proxies do not prove improved retention.
